# OASIS-3 External Validation for Scopus Paper 1 (share-nothing workflow)

Everything runs in **your own Colab and Google Drive**. Nothing from OASIS-3 is ever shared outside them.

**Part A – prepare data** (cells A1–A6): select subjects, download T1 MRI, preprocess to match OASIS-1, save slices.
**Part B – validate** (cells B1–B3): **Runtime → Restart session first**, then test the 5 saved Paper 1 models.

Before starting, add two **Colab Secrets** (🔑 key icon on the left): `NITRC_USER` = your NITRC username, `NITRC_PASS` = your NITRC password. Turn on *Notebook access* for both. Never type the password into a cell.

All rules (model, cut-offs, stage definitions, subject selection) are fixed **before** any OASIS-3 result is seen.

In [ ]:
# A1. Setup (Part A)
!pip -q install antspyx antspynet nibabel
import os, glob, re, io, gzip, json, zipfile, time, gc
import numpy as np, pandas as pd, nibabel as nib, cv2, requests
import matplotlib.pyplot as plt
import ants
from google.colab import drive, userdata
drive.mount('/content/drive')
print('ANTsPy', ants.__version__)

In [ ]:
# A2. Settings (fixed BEFORE seeing any OASIS-3 result)
CLIN    = '/content/drive/MyDrive/OASIS3/clinical'          # uploaded OASIS3_data_files (private)
O3      = '/content/drive/MyDrive/OASIS3'
RAW     = f'{O3}/raw_T1'                                    # downloaded T1 scans
SLC     = f'{O3}/slices'                                    # 25 x 128 x 128 slices per subject
QC      = f'{O3}/qc'
OASIS1  = '/content/drive/MyDrive/OASIS1_masked'            # used only to build the OASIS-1 template
RESULTS = '/content/drive/MyDrive/OASIS1_results'
for d in (RAW, SLC, QC): os.makedirs(d, exist_ok=True)

IMG, N_SLICES, STEP = 128, 25, 2                            # identical to OASIS-1
CENTRE   = int(open(f'{RESULTS}/centre.txt').read()) if os.path.exists(f'{RESULTS}/centre.txt') else 103
MAX_DAYS = 180      # MRI must be within 180 days of a CDR assessment
PER_STAGE = 80      # subjects per stage (all of them if fewer are available)
SEED = 42
print('Coronal centre index:', CENTRE)

In [ ]:
# A3. Select ONE MRI session per subject, paired with the closest CDR assessment (prints only counts and column NAMES)
LABEL = re.compile(r'(OAS3\d{4})_[A-Za-z0-9]+_d(\d{4,5})')
MRLAB = re.compile(r'(OAS3\d{4}_MR_d\d{4,5})')

def find_csv(key):
    hits = [f for f in glob.glob(f'{CLIN}/**/*.csv', recursive=True) if key.lower() in f.lower()]
    assert hits, f'No CSV containing "{key}" found under {CLIN}'
    return sorted(hits)[0]

def label_column(d, pattern=LABEL):
    for c in d.columns:
        if d[c].astype(str).str.contains(pattern).mean() > 0.9: return c
    raise ValueError('No column with OASIS-3 visit labels found')

cdr_raw = pd.read_csv(find_csv('UDSb4'), low_memory=False)
mr_raw  = pd.read_csv(find_csv('MRI-json'), low_memory=False)

# global CDR column: name contains "cdr" and values only in {0, 0.5, 1, 2, 3}
def only_cdr_values(c):
    v = pd.to_numeric(cdr_raw[c], errors='coerce').dropna()
    return len(v) > 0 and set(v.unique()) <= {0, 0.5, 1, 2, 3}
cands = [c for c in cdr_raw.columns if 'cdr' in c.lower() and only_cdr_values(c)]
preferred = [c for c in cands if c.lower() in ('cdrtot', 'cdr', 'cdrglob', 'cdr_global', 'cdrglobal')]
assert cands, 'No global CDR column found - tell Claude: "no CDR column found"'
CDR_COL = (preferred or cands)[0]
print('Candidate CDR columns:', cands, '| USING:', CDR_COL, '<- check this is the GLOBAL CDR')

lc = label_column(cdr_raw)
cdr = cdr_raw[[lc, CDR_COL]].copy()
m = cdr[lc].astype(str).str.extract(LABEL)
cdr['subject'], cdr['day'] = m[0], pd.to_numeric(m[1])
cdr['cdr'] = pd.to_numeric(cdr[CDR_COL], errors='coerce')
cdr = cdr.dropna(subset=['subject', 'day', 'cdr'])

mlc = label_column(mr_raw, MRLAB)
mr = pd.DataFrame({'session': mr_raw[mlc].astype(str).str.extract(MRLAB)[0]}).dropna().drop_duplicates()
m = mr['session'].str.extract(LABEL)
mr['subject'], mr['day'] = m[0], pd.to_numeric(m[1])

pairs = mr.merge(cdr[['subject', 'day', 'cdr']], on='subject', suffixes=('_mr', '_cdr'))
pairs['gap'] = (pairs['day_mr'] - pairs['day_cdr']).abs()
pairs = pairs[pairs['gap'] <= MAX_DAYS].sort_values(['subject', 'gap', 'day_mr'])
best = pairs.groupby('subject').head(1).copy()               # closest CDR -> one session per subject
best['stage'] = np.select([best.cdr == 0, best.cdr == 0.5, best.cdr >= 1], [0, 1, 2])

sel = pd.concat([g.sample(min(len(g), PER_STAGE), random_state=SEED) for _, g in best.groupby('stage')])
sel = sel.sort_values('session').reset_index(drop=True)
sel.to_csv(f'{O3}/selection.csv', index=False)               # stays in YOUR Drive
print('Eligible subjects per stage [ND, VM, Mild/Mod]:', np.bincount(best.stage, minlength=3))
print('Selected subjects per stage [ND, VM, Mild/Mod]:', np.bincount(sel.stage, minlength=3), '| total', len(sel))

In [ ]:
# A4. Download the selected T1 scans from NITRC-IR (credentials from Colab Secrets; re-running skips finished files)
USER, PWD = userdata.get('NITRC_USER'), userdata.get('NITRC_PASS')
BASE = 'https://www.nitrc.org/ir'
sess = requests.Session()

def login():
    r = sess.post(f'{BASE}/data/JSESSION', auth=(USER, PWD), timeout=60)
    assert r.ok and len(r.text) > 10, f'NITRC login failed (HTTP {r.status_code}) - check the Colab Secrets'
login()

def get(url, **kw):
    r = sess.get(url, timeout=900, **kw)
    if r.status_code in (401, 403):                           # session expired -> log in again once
        login(); r = sess.get(url, timeout=900, **kw)
    r.raise_for_status(); return r

def download_t1(label):
    out = f'{RAW}/{label}.nii.gz'
    if os.path.exists(out): return 'exists'
    exp = f'{BASE}/data/projects/OASIS3/experiments/{label}'
    scans = get(f'{exp}/scans', params={'format': 'json'}).json()['ResultSet']['Result']
    t1 = [s['ID'] for s in scans if 'T1w' in str(s.get('type', ''))]
    if not t1: return 'no T1w'
    z = zipfile.ZipFile(io.BytesIO(get(f'{exp}/scans/{t1[0]}/resources/NIFTI/files', params={'format': 'zip'}).content))
    nii = [n for n in z.namelist() if n.endswith(('.nii.gz', '.nii'))]
    if not nii: return 'no NIfTI'
    data = z.read(nii[0])
    open(out, 'wb').write(data if nii[0].endswith('.gz') else gzip.compress(data))
    return 'downloaded'

sel = pd.read_csv(f'{O3}/selection.csv')
status = {}
for i, label in enumerate(sel.session):
    try:
        status[label] = download_t1(label)
    except Exception as e:
        status[label] = f'error: {type(e).__name__}'
    if (i + 1) % 20 == 0: print(f'{i + 1}/{len(sel)} processed')
pd.Series(status).to_csv(f'{O3}/download_status.csv')
print(pd.Series(status).str.split(':').str[0].value_counts())

In [ ]:
# A5. Build the OASIS-1 template and fix the orientation (uses 3 OASIS-3 subjects; ~10-20 min, done once)
import antspynet
sel = pd.read_csv(f'{O3}/selection.csv')

def load_o1(path):                                    # identical to the OASIS-1 pipeline
    v = np.squeeze(np.asarray(nib.load(path).dataobj)).astype(np.float32)
    v /= np.percentile(v[v > 0], 99.5); np.clip(v, 0, 1, out=v); return v

tfile = f'{O3}/oasis1_template.npy'
if not os.path.exists(tfile):
    vols = sorted(glob.glob(f'{OASIS1}/*_mpr_n*_anon_111_t88_masked_gfc.img'))[:40]
    np.save(tfile, np.mean([load_o1(p) for p in vols], axis=0).astype(np.float32))
T = np.load(tfile); TPL = ants.from_numpy(T)
print('Template shape:', T.shape)

def brain_array(path):
    img = ants.n4_bias_field_correction(ants.image_read(path), shrink_factor=4)
    prob = antspynet.brain_extraction(img, modality='t1', verbose=False)
    brain = ants.reorient_image2(img * ants.threshold_image(prob, 0.5, 1.0), 'RAS')
    return brain.numpy().astype(np.float32), brain.spacing

def register(a, spacing, flips):
    for ax, f in enumerate(flips):
        if f: a = np.flip(a, axis=ax)
    mov = ants.from_numpy(np.ascontiguousarray(a), spacing=spacing)
    w = ants.registration(fixed=TPL, moving=mov, type_of_transform='Affine')['warpedmovout'].numpy()
    corr = np.corrcoef(w[T > 0.05].ravel(), T[T > 0.05].ravel())[0, 1]
    return w, corr

ofile = f'{O3}/orientation.json'
if not os.path.exists(ofile):
    ok_files = [f'{RAW}/{s}.nii.gz' for s in sel.session if os.path.exists(f'{RAW}/{s}.nii.gz')][:3]
    combos = [(a, b, c) for a in (0, 1) for b in (0, 1) for c in (0, 1)]
    score = {c: [] for c in combos}
    for f in ok_files:
        a, sp = brain_array(f)
        for c in combos: score[c].append(register(a.copy(), sp, c)[1])
    FLIPS = max(combos, key=lambda c: np.mean(score[c]))
    json.dump({'flips': FLIPS, 'corr': {str(c): float(np.mean(v)) for c, v in score.items()}}, open(ofile, 'w'))
FLIPS = tuple(json.load(open(ofile))['flips'])
print('Orientation fixed: flips =', FLIPS, '| mean correlation per option:', json.load(open(ofile))['corr'])

In [ ]:
# A6. Preprocess every downloaded scan -> 25 coronal slices (same as OASIS-1). ~1-2 min per scan; resumes after a disconnect
def coronal_slices(v, centre):                        # identical to the OASIS-1 pipeline
    ap = int(np.argmax(v.shape)); out = []
    for k in range(N_SLICES):
        idx = centre + (k - N_SLICES // 2) * STEP
        sl = np.ascontiguousarray(np.rot90(np.take(v, idx, axis=ap)))
        out.append(cv2.resize(sl, (IMG, IMG), interpolation=cv2.INTER_AREA))
    return np.stack(out).astype(np.float16)

qc = {}
qfile = f'{O3}/registration_qc.csv'
if os.path.exists(qfile): qc = pd.read_csv(qfile, index_col=0)['corr'].to_dict()
todo = [s for s in sel.session if os.path.exists(f'{RAW}/{s}.nii.gz') and not os.path.exists(f'{SLC}/{s}.npy')]
print(len(todo), 'scans to preprocess')
for i, s in enumerate(todo):
    try:
        a, sp = brain_array(f'{RAW}/{s}.nii.gz')
        w, corr = register(a, sp, FLIPS)
        w[w < 0] = 0; w /= np.percentile(w[w > 0], 99.5); np.clip(w, 0, 1, out=w)
        np.save(f'{SLC}/{s}.npy', coronal_slices(w, CENTRE)); qc[s] = corr
    except Exception as e:
        qc[s] = np.nan; print(s[:9] + '...', 'failed:', type(e).__name__)
    if (i + 1) % 10 == 0:
        pd.Series(qc, name='corr').to_csv(qfile); print(f'{i + 1}/{len(todo)} done')
    gc.collect()
pd.Series(qc, name='corr').to_csv(qfile)
c = pd.Series(qc).dropna()
print(f'Preprocessed: {len(c)} | registration correlation median {c.median():.2f}, '
      f'min {c.min():.2f} | below 0.5 (check these): {(c < 0.5).sum()}')

# Quality-control picture (shown on YOUR screen only): template vs 8 registered subjects, middle slice
done = [s for s in sel.session if os.path.exists(f'{SLC}/{s}.npy')][:8]
fig, ax = plt.subplots(1, 9, figsize=(18, 2.4))
ax[0].imshow(coronal_slices(T, CENTRE)[N_SLICES // 2].astype(np.float32), cmap='gray'); ax[0].set_title('OASIS-1 template')
for i, s in enumerate(done):
    ax[i + 1].imshow(np.load(f'{SLC}/{s}.npy')[N_SLICES // 2].astype(np.float32), cmap='gray'); ax[i + 1].set_title(f'subject {i + 1}')
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig(f'{QC}/registration_check.png', dpi=120); plt.show()
print('Check: every subject should look like the template (same orientation, brain centred, ventricles visible).')

## Part B – External validation
**First: Runtime → Restart session.** Then run B1–B3 (no GPU needed). The model, the cut-offs and the decision rule below were fixed before any OASIS-3 result was seen.

In [ ]:
# B1. Setup (Part B, after restarting the session)
import os, json, numpy as np, pandas as pd
import keras
from google.colab import drive
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, cohen_kappa_score,
                             mean_absolute_error, confusion_matrix)
drive.mount('/content/drive')
O3      = '/content/drive/MyDrive/OASIS3'
MODELS  = '/content/drive/MyDrive/OASIS1_results/paper1/exp5_gradcam'      # the 5 saved final Paper 1 models
OUT     = '/content/drive/MyDrive/OASIS1_results/paper1/exp7_external_oasis3'
os.makedirs(OUT, exist_ok=True)

# Pre-specified decision rule (from OASIS-1 only):
CUTS   = [(1.45, 1.60), (0.65, 1.45), (1.05, 1.30), (0.85, 1.35), (0.75, 1.50)]   # validation cut-offs of folds 0-4 (Cell 14)
T1, T2 = float(np.median([c[0] for c in CUTS])), float(np.median([c[1] for c in CUTS]))
print(f'Ensemble rule: mean disease score of the 5 models, cut-offs {T1:.2f} / {T2:.2f}')

In [ ]:
# B2. Predict with the 5 models (no training, no tuning on OASIS-3)
sel = pd.read_csv(f'{O3}/selection.csv')
sel = sel[[os.path.exists(f'{O3}/slices/{s}.npy') for s in sel.session]].reset_index(drop=True)
X3 = np.stack([np.load(f'{O3}/slices/{s}.npy') for s in sel.session]).astype(np.float32)
y3 = sel.stage.values.astype(int); n, ns = X3.shape[:2]
print('External test set:', n, 'subjects | [ND, VM, Mild/Mod] =', np.bincount(y3, minlength=3))

apply_cuts = lambda s, a, b: np.where(s < a, 0, np.where(s < b, 1, 2))
scores = np.zeros((n, 5))
for k in range(5):
    m = keras.models.load_model(f'{MODELS}/model_fold{k}.keras')
    P = m.predict(X3.reshape(-1, 128, 128, 1), batch_size=256, verbose=0)
    scores[:, k] = P.reshape(n, ns, 2).sum(axis=2).mean(axis=1)
    print(f'model {k} done')
np.save(f'{OUT}/scores.npy', scores)

In [ ]:
# B3. External-validation results (summary numbers only - these may be shared)
mf1 = lambda y, p: f1_score(y, p, average='macro', zero_division=0)
def summary(y, p):
    cm = confusion_matrix(y, p, labels=[0, 1, 2]); rec = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
    return {'acc': accuracy_score(y, p), 'bal_acc': balanced_accuracy_score(y, p), 'macro_F1': mf1(y, p),
            'QWK': cohen_kappa_score(y, p, weights='quadratic'), 'MAE': mean_absolute_error(y, p),
            'ND_rec': rec[0], 'VM_rec': rec[1], 'MM_rec': rec[2]}

ens = apply_cuts(scores.mean(axis=1), T1, T2)                                   # PRIMARY result
rng = np.random.default_rng(42); boot = [rng.integers(0, n, n) for _ in range(2000)]
ci = lambda f: np.percentile([f(y3[i], ens[i]) for i in boot], [2.5, 97.5])
res = summary(y3, ens)
print('===== PRIMARY: 5-model ensemble, pre-specified cut-offs =====')
for k, v in res.items(): print(f'{k:9s} {v:.3f}')
for name, f in [('bal_acc', balanced_accuracy_score), ('macro_F1', mf1), ('QWK', lambda a, b: cohen_kappa_score(a, b, weights="quadratic"))]:
    lo, hi = ci(f); print(f'{name} 95% CI [{lo:.3f}, {hi:.3f}]')
print('\nConfusion matrix (rows = true ND/VM/MM, columns = predicted):')
print(confusion_matrix(y3, ens, labels=[0, 1, 2]))

per = pd.DataFrame([summary(y3, apply_cuts(scores[:, k], *CUTS[k])) for k in range(5)])
print('\n===== SECONDARY: each fold model with its own cut-offs (mean +/- SD of 5 models) =====')
print(per.agg(['mean', 'std']).round(3).to_string())
pd.DataFrame([res]).to_csv(f'{OUT}/external_primary.csv', index=False); per.to_csv(f'{OUT}/external_per_model.csv', index=False)